In [1]:
import json
import math
import random
import torch


# train_data = MyLLMDataloader(4, tokenizer, "cleaned_TeleQnA_train_context_gte.json", shuffle=True)
import pandas as pd
import os

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm


import matplotlib.pyplot as plt
import pandas as pd
import itertools
import time
from statistics import mode
from transformers import logging

logging.set_verbosity_error()



In [2]:
orig_test_data = []
with open("dev.json", "r") as f:
    test_data = f.readlines()

for line in test_data:
    orig_test_data.append(json.loads(line))


In [3]:
len(orig_test_data)

4183

In [4]:
# Function to clean and split text into words
def preprocess(text):
    
    text = text.lower()
    return set(text.split())


def choose_most_likely_option(options, candidate_answer):
    candidate_words = preprocess(candidate_answer)
    
    best_option = None
    max_overlap = 0
    
    for option in options:
        option_words = preprocess(option)
        overlap = len(candidate_words.intersection(option_words))
        
  
        if overlap > max_overlap:
            max_overlap = overlap
            best_option = option
            
    return best_option, options.index(best_option)+1

In [5]:
from string import Template
prompt_q_without_contex_train= Template('''Instruct = Youre a Medical Question Answering Expert, answer the following question. Please generate only answer choice (1, 2, 3 or 4)\n
$question
$options
$question
''')


prompt_without_contex_train= Template('''Instruct = Youre a Medical Question Answering Expert, answer the following question. Please generate only answer choice (1, 2, 3 or 4)\n                                                                 
$question
$options
Output: option ''')




cache_prompt= Template('''Instruct : Youre a Medical Question Answering Expert, answer the following question. Please generate only answer choice (1, 2, 3 or 4)\n                                                  
$question
''')

option_prompt = Template('''
$options
Output: option ''')
# prompt_without_context = f'Hello {planet}'

In [6]:
def clean_question(question):
    for num in [14, 15, 16, 17, 18]:
        question = question.replace(f"[3GPP Release {num}]", "")
    return question

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
BASE_MODEL_ID = "microsoft/phi-2"
BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
device = "auto"
# torch.set_default_device(device)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, trust_remote_code=True, device_map="auto")
# model.to(device)


Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.73s/it]


In [8]:
data = []
with open("dev.json", "r") as f:
    test_data = f.readlines()

for line in test_data:
    data.append(json.loads(line))

# data[0]

option_header = ["option 1 ", "option 2 ", "option 3 ", "option 4 ", "option 5 "]

example = data[0]
options = []
opts = []
i =  0 
for key in example.keys():
    # print(key)

    if key.startswith("op"):
        if example[key] == None:
            continue
        options.append(option_header[i]+ example[key])
        opts.append((example[key], key.split("op")[1]))
        i+=1

instruct = "Youre a Medical Question Answering Expert, answer the following question. Please generate only answer choice (1, 2, 3 or 4)\n"
prompt_sample = (instruct+ example["question"] +"\n"+ "\n".join(options))

print(prompt_sample)

input  = tokenizer(prompt_sample, return_tensors="pt")
# print(input)
out = model.generate(**input,  max_length=200)
text = tokenizer.batch_decode(out)[0]
print(text)

Youre a Medical Question Answering Expert, answer the following question. Please generate only answer choice (1, 2, 3 or 4)
Which of the following is not true for myelinated nerve fibers:
option 1 Impulse through myelinated fibers is slower than non-myelinated fibers
option 2 Membrane currents are generated at nodes of Ranvier
option 3 Saltatory conduction of impulses is seen
option 4 Local anesthesia is effective only when the nerve is not covered by myelin sheath


/opt/conda/lib/python3.12/site-packages/transformers/generation/utils.py:1934: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


Youre a Medical Question Answering Expert, answer the following question. Please generate only answer choice (1, 2, 3 or 4)
Which of the following is not true for myelinated nerve fibers:
option 1 Impulse through myelinated fibers is slower than non-myelinated fibers
option 2 Membrane currents are generated at nodes of Ranvier
option 3 Saltatory conduction of impulses is seen
option 4 Local anesthesia is effective only when the nerve is not covered by myelin sheath

The correct answer is: 1<|endoftext|>


In [9]:
class MyLLMDataloader:
    def __init__(self, batch_size, tokenizer, data, shuffle = False, val= False, k=20):
        ## initializations
        self.batch_size  = batch_size
        self.tokenizer  = tokenizer
        self.tokenizer.pad_token = self.tokenizer.eos_token
        # with open(data, "r") as f:
        #     self.data = json.load(f)
        self.data = []
        with open(data, "r") as f:
            test_data = f.readlines()

        for line in test_data:
            self.data.append(json.loads(line))

        # self.all_examples = list(self.data.keys())
        self.shuffle = shuffle
        self.val = val
        self.k = k
        self.n_data_points = math.ceil(len(self.data)/self.batch_size)
        self.indices = [i for i in range(self.n_data_points)]
        
    def __getitem__(self, idx):
        ## this gets a batch 
        option_header = ["option 1 ", "option 2 ", "option 3 ", "option 4 ", "option 5 "]
        batch_start_id = idx * self.batch_size
        mapper_ans = {"a":1, "b":2, "c":3, "d":4, "e":5}
        batch_end_id  = min(len(self.data), batch_start_id + self.batch_size) 
        batch = {"options":[], "answer":[], "question_context":[],}
        
        batch_type =False
        for i in range(batch_start_id, batch_end_id):
            example = self.data[i]
            options = []
            opts = []
            for key in example.keys():
                if key.startswith("op"):
                    if example[key] == None:
                        continue
                    options.append(example[key])
                    opts.append((example[key], mapper_ans[key.split("op")[1]] ))
            
            string_opts = ' '.join(opt[0] for opt in opts)
            batch_prompts = []
            option_maps = []
         
            if not ("option" in string_opts or "above" in string_opts) and self.k>1:
                    batch_type = True
                    question_context = cache_prompt.substitute(question = clean_question(example["question"]))
                    batch["question_context"].append(question_context)

                    all_permutations = list(itertools.permutations(opts))
                    all_permutations = random.sample(all_permutations, self.k if len(all_permutations)>self.k else len(all_permutations))
                    
                    for option_set in all_permutations:
                        option_map = []
                        options_with_header   = []
                        for z in range(len(option_set)):

                            options_with_header.append(option_header[z] +option_set[z][0])

                         
                            option_map.append(int(option_set[z][1]))
                        
                        
                        
                        options_with_header = "\n".join(options_with_header)

                        prompt= option_prompt.substitute(options =options_with_header)
                        # print(question_context, prompt)
                        batch_prompts.append(prompt)
                        option_maps.append(option_map)


            else:
                options_with_header = [option_header[i] +options[i] for i in range(len(options)) ]
               
                options_with_header = "\n".join(options_with_header)
                prompt = prompt_without_contex_train.substitute(question = clean_question(example["question"]), options =options_with_header)
            
                batch_prompts.append(prompt)
                

                


            batch["options"] += batch_prompts
            
            # batch["answer"] += [answer]

        self.tokenizer.padding_side = "left"
        question_context_tokens =  {"input_ids":None,"attention_mask":None}
        q_tokens = self.tokenizer(batch["options"], padding="longest", return_tensors="pt")  
        if batch_type:
            # print("orig mask", q_tokens["attention_mask"].shape, q_tokens["attention_mask"])
            question_context_tokens = self.tokenizer(batch["question_context"], padding="longest", return_tensors="pt")  
            attn_masks = torch.ones((q_tokens["attention_mask"].shape[0], q_tokens["attention_mask"].shape[1] + len(question_context_tokens[0])))
        else:
            attn_masks = q_tokens["attention_mask"]
        self.tokenizer.padding_side = "right"
        # a_tokens = self.tokenizer(batch["answer"], padding="longest", return_tensors="pt")
        tokens = q_tokens
        
       

        # attn_masks = torch.cat([q_tokens["attention_mask"], a_tokens["attention_mask"]], dim=1)
        # loss_mask = torch.cat([torch.zeros_like(q_tokens["attention_mask"]), a_tokens["attention_mask"]], dim=1)[:,1:]
   
        result = {
        "inp_ids":tokens["input_ids"],
        "inp_mask":attn_masks,## Causal Training
        "option_maps": option_maps,
        "cached": batch_type,
        "question_context_ids":question_context_tokens["input_ids"],
        "question_mask":question_context_tokens["attention_mask"],


        }

        # result["loss_mask"] = loss_mask * result["out_mask"]
        # result["out_ids"][:,:q_tokens["input_ids"].size(1)-10] = self.tokenizer.eos_token_id

        return result       


            

    def __iter__(self):
        self.idx = 0
        return self

    def __next__(self):
        if self.idx >= self.n_data_points:
            self.idx = 0
            raise StopIteration
        temp_idx = self.indices[self.idx]
        self.idx += 1
        return self[temp_idx]
             








            




    
    def __len__(self):
        return self.n_data_points
    



        

In [10]:
def forward_pass(model, batch, k=5):
    inp_ids = batch["inp_ids"].to(model.device)
    attn_mask = batch["inp_mask"].to(model.device)

    if batch["cached"]:
        with torch.inference_mode():
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                prompt_cache = model(input_ids=batch["question_context_ids"], attention_mask=batch["question_mask"], use_cache = True).past_key_values
                # Convert prompt_cache to a list if it's a tuple
                prompt_cache = list(prompt_cache)
                qc_len = prompt_cache[0][0].shape[2]
                with open('tokenssavings.txt', 'a') as the_file:
                    the_file.write(str((k-1)*qc_len/ (k *attn_mask.shape[1])) +'\n')
              
                for i, (ke, v) in enumerate(prompt_cache):
                
                    k_repeated = ke.repeat(k, 1, 1, 1)
                    # print(ke.shape, k_repeated.shape)
                    v_repeated = v.repeat(k, 1, 1, 1)

        
                    prompt_cache[i] = (k_repeated, v_repeated)

        
                prompt_cache = tuple(prompt_cache)



                
                result = model(input_ids=inp_ids, attention_mask=attn_mask, past_key_values=prompt_cache)
        return result.logits

    result = model(input_ids=inp_ids, attention_mask=attn_mask)
    logits = result.logits
    return logits

In [11]:
def inference(model, testLoader, k = 15):
    my_ans = {"Answer_ID": []}
 
    model.eval()          
    option_ids = [tokenizer(o).input_ids[0] for o in ["1", "2", "3", "4", "5"]]
    pbar = tqdm(range(len(testLoader)), )
    for item in testLoader:
        
        # if int(tokenizer.decode(item["a_tokens"].input_ids[:,0], skip_special_tokens=True)) ==0:
        if len(item["option_maps"]) >0:
            # print('\n'.join(tokenizer.batch_decode(item['inp_ids']))),
            # first_half = item.copy()

        




            # print(first_half["inp_ids"].shape, second_half["inp_ids"].shape)
            with torch.inference_mode():
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    # gen_tokens = model.generate(inputs=item["inp_ids"].to(device), max_new_tokens=1)

                    start = time.time()
                    logits = forward_pass(model, item,k=item["inp_ids"].shape[0])
                    with open('time_bqkv.txt', 'a') as the_file:
                            the_file.write(str(time.time()- start) +'\n')
                        # logits2 = forward_pass(model, second_half)
                  
                
            
  
            # print(logits.shape)
            
            preds =(logits[:, -1, option_ids ].argmax(dim=1) ).cpu()
            z = torch.tensor(item["option_maps"])
            # print("predictions",preds, z)
         
            print( tokenizer.batch_decode(item["inp_ids"]),"predicted", preds.item(), tokenizer.batch_decode(logits.argmax(axis=2), skip_special_tokens=True)[0])
            preds = torch.mode(z.gather(1, preds.unsqueeze(1)).squeeze(1)).values
            my_ans["Answer_ID"].append(preds.item())
            # print("logits", tokenizer.batch_decode(gen_tokens,  skip_special_tokens=True)[0] )
            # print(tokenizer.decode(item["a_tokens"].input_ids[:,0], skip_special_tokens=True))
            # break
        else:
            with torch.inference_mode():
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    # gen_tokens = model.generate(inputs=item["inp_ids"].to(device), max_new_tokens=1)
                    start = time.time()
                    logits = forward_pass(model, item, k =1)
                    with open('time_bqkv.txt', 'a') as the_file:
                        the_file.write(str(time.time()- start) +'\n')
                    preds =(logits[:, -1, option_ids ].argmax(dim=1) +1)


                    my_ans["Answer_ID"].append(preds.item())
        
        pd.DataFrame(my_ans).to_csv("medmcq_ans.csv")
        pbar.set_description(f"Prediction: {preds}")
        pbar.update(1)
    return my_ans

    

In [12]:
# k = 15
# testLoader = MyLLMDataloader(1, tokenizer, "questions_new_final_backup.json", val=True, shuffle=False, k=k)
# model.eval() 
# i = 0
# for dat in testLoader:
#     if i ==1:
#         start = time.time()
    
#         logit = forward_pass(model, dat,k=k)
#         print(time.time()-start)

#         break
#     i+=1

In [13]:
k=1
testLoader = MyLLMDataloader(1, tokenizer, "dev.json", val=True, shuffle=False, k=k)
for doc in testLoader:
    break

In [14]:
k = 1
testLoader = MyLLMDataloader(1, tokenizer, "dev.json", val=True, shuffle=False, k=k)

predictions = inference(model, testLoader,  k =k)

pd.DataFrame(predictions).to_csv("medmcq_qwen3B_ans_k_"+ str(k)+".csv")

Prediction: tensor([1], device='cuda:0'):   0%|          | 1/4183 [00:00<10:28,  6.66it/s]

Prediction: tensor([1], device='cuda:0'): 100%|██████████| 4183/4183 [05:45<00:00, 12.11it/s]


In [15]:
submission_df = pd.read_csv('qwen2.5-3B-instruct_inference_trick.csv')
submission_df["Answer_ID"] = predictions["Answer_ID"]
submission_df.to_csv("law_ans_k_"+ str(k)+".csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'qwen2.5-3B-instruct_inference_trick.csv'

In [ ]:
def compute_mean_from_file(file_path):
    # Read the file and extract the values
    with open(file_path, 'r') as file:
        values = [float(line.strip()) for line in file]
    
    # Compute the mean
    mean_value = sum(values) / len(values) if values else 0
    return mean_value

# Example usage
file_path = 'time_bqkv.txt'  # Replace with your actual file path
mean = compute_mean_from_file(file_path)
print(f"The mean is: {mean}")

In [ ]:
import pandas as pd
# Load the CSV files into DataFrames
answers_df = pd.read_csv('answers.csv')
submission_df = pd.read_csv('law_ans.csv').iloc[:366]
submission_df["Question_ID"] = answers_df["Question_ID"]
print(len(answers_df), len(submission_df))
# Merge the DataFrames on the 'Question_ID' column
merged_df = pd.merge(answers_df, submission_df, on='Question_ID', suffixes=('_correct', '_submitted'))

# Calculate the number of correct answers
correct_answers = (merged_df['Answer_ID_correct'] == merged_df['Answer_ID_submitted']).sum()

# Calculate the total number of questions
total_questions = len(merged_df)

# Calculate accuracy
accuracy = correct_answers / total_questions

print(f'Accuracy: {accuracy:.2%}')

In [34]:
answers_df = pd.read_csv('law_ans_best.csv')
submission_df = pd.read_csv('law_ans.csv')

In [35]:
answers_df['Answer_ID'][:len(submission_df)] = submission_df['Answer_ID']

In [36]:

answers_df.to_csv("law_ans_bqkv.csv",index=False)